# CALCE CS2 Dataset — Exploratory Data Analysis

This notebook explores the CALCE Center for Advanced Life Cycle Engineering
battery aging dataset (CS2_33, CS2_34, CS2_35, CS2_36). We examine:
1. Dataset inventory and cycle structure
2. Discharge capacity fade curves
3. Voltage and temperature evolution across aging
4. SOH label computation and distribution

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.preprocessing.data_loader import (
    load_all_calce_cells, CALCE_RATED_CAPACITY, CALCE_CUTOFF_VOLTAGE,
)
from src.preprocessing.soh import compute_soh_for_all_cells

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100

## 1. Load Data

In [ ]:
cells = load_all_calce_cells("../data/raw/calce")
print(f"Loaded {len(cells)} cells")

for cell_id, cell_data in cells.items():
    n_charge = sum(1 for c in cell_data["cycles"] if c["type"] == "charge")
    n_discharge = sum(1 for c in cell_data["cycles"] if c["type"] == "discharge")
    print(f"  {cell_id}: {n_charge} charge, {n_discharge} discharge")
    print(f"    Rated capacity: {cell_data['rated_capacity']} Ah, Cutoff: {cell_data['cutoff_voltage']} V")

## 2. Inventory Table

In [ ]:
records = []
for cell_id, cell_data in cells.items():
    discharge_cycles = [c for c in cell_data["cycles"] if c["type"] == "discharge"]
    capacities = [c["capacity"] for c in discharge_cycles if c["capacity"] is not None]
    records.append({
        "Cell ID": cell_id,
        "Rated Capacity (Ah)": cell_data["rated_capacity"],
        "Cutoff Voltage (V)": cell_data["cutoff_voltage"],
        "Charge Cycles": sum(1 for c in cell_data["cycles"] if c["type"] == "charge"),
        "Discharge Cycles": len(discharge_cycles),
        "Max Capacity (Ah)": float(np.max(capacities)) if capacities else np.nan,
        "Min Capacity (Ah)": float(np.min(capacities)) if capacities else np.nan,
    })

inventory_df = pd.DataFrame(records)
inventory_df

## 3. Discharge Capacity Fade Curves

In [ ]:
fig, ax = plt.subplots()

for cell_id, cell_data in cells.items():
    discharge_cycles = [c for c in cell_data["cycles"] if c["type"] == "discharge"]
    cycle_nums = [c["cycle_number"] for c in discharge_cycles]
    capacities = [c["capacity"] for c in discharge_cycles if c["capacity"] is not None]
    ax.plot(cycle_nums[:len(capacities)], capacities, label=cell_id, linewidth=1.5)

ax.axhline(y=CALCE_RATED_CAPACITY * 0.80, color="red", linestyle="--",
           alpha=0.7, label=f"EOL (80% of {CALCE_RATED_CAPACITY} Ah)")
ax.set_xlabel("Cycle Number")
ax.set_ylabel("Discharge Capacity (Ah)")
ax.set_title("CALCE CS2 — Discharge Capacity Fade")
ax.legend()
plt.tight_layout()
plt.show()

## 4. Voltage Curve Evolution

In [ ]:
cell_id = "CS2_33"
cell_data = cells[cell_id]
discharge_cycles = [c for c in cell_data["cycles"] if c["type"] == "discharge"]

# Select curves at different aging stages
n = len(discharge_cycles)
stages = [0, n // 4, n // 2, 3 * n // 4, n - 1]
stage_labels = ["0%", "25%", "50%", "75%", "100%"]

fig, ax = plt.subplots()
for idx, label in zip(stages, stage_labels):
    dc = discharge_cycles[idx]
    ax.plot(dc["voltage"], dc["current"], label=f"{label} of life (cycle {dc['cycle_number']})", linewidth=1.5)

ax.set_xlabel("Voltage (V)")
ax.set_ylabel("Current (A)")
ax.set_title(f"{cell_id} — Discharge Voltage-Current Curves at Different Aging Stages")
ax.legend()
plt.tight_layout()
plt.show()

## 5. SOH Label Computation

In [ ]:
soh_df = compute_soh_for_all_cells(cells)
print(f"SOH labels: {len(soh_df)} rows, {soh_df['cell_id'].nunique()} cells")
soh_df.head(10)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# SOH curves
for cell_id in soh_df["cell_id"].unique():
    cell_soh = soh_df[soh_df["cell_id"] == cell_id]
    axes[0].plot(cell_soh["cycle_number"], cell_soh["soh"], label=cell_id, linewidth=1.5)
axes[0].set_xlabel("Cycle Number")
axes[0].set_ylabel("SOH")
axes[0].set_title("SOH vs. Cycle Number")
axes[0].legend()

# SOH distribution
axes[1].hist(soh_df["soh"], bins=50, edgecolor="black", alpha=0.7)
axes[1].set_xlabel("SOH")
axes[1].set_ylabel("Count")
axes[1].set_title("SOH Distribution")

plt.tight_layout()
plt.show()

## 6. Save SOH Labels

In [ ]:
soh_df.to_parquet("../data/processed/soh_labels_calce.parquet", index=False)
print(f"Saved SOH labels to data/processed/soh_labels_calce.parquet ({len(soh_df)} rows)")